# 05 · Build robustness (obfuscation) test sets
**Purpose:** create transformed versions of the **test split** once, so every detector scores identical attacked texts.

| ID | Transformation | Applied to | Levels | Evidence for inclusion |
|---|---|---|---|---|
| T1 | LLM paraphrase (Qwen3-235B, a family *not* used for generation) | machine | 1 | most destructive for neural detectors (DetectRL-X, ACL 2026); multilingual ChatGPT paraphrase in Macko et al. 2024 |
| T2 | Back-translation via NLLB-200 (pivot en; ru for English) | machine | 1 | Macko et al. 2024; DetectRL |
| T3 | Homoglyph Cyrillic↔Latin | machine, and machine+human ("both") | 5%, 15% of eligible chars | most damaging attack in Macko et al. 2024, RAID, SilverSpeak |
| T4 | Character typos | machine, and both | 5%, 15% of words | RAID; DetectRL |

Plus the **defence** variants (`*_def`): Unicode NFKC + confusable mapping applied before scoring.
Quality control: LaBSE cosine similarity to the original and GlotLID language check (T1, T2).

**Outputs:** `05_attacks/attacks_v1.parquet`, `05_attacks/attack_quality.csv`, caches. **Runtime:** GPU (T4/L4) ~30–60 min; OpenRouter ≈ $1–3.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "05_build_attack_sets"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
!pip -q install fasttext sentence-transformers sacremoses

In [ ]:
A = CFG["attacks"]; D = CFG["data"]
ds = load_dataset()
test = ds[ds.split == "test"].reset_index(drop=True)
mach = test[test.label == 1]; hum = test[test.label == 0]
log.info(f"test docs: {len(test)} (machine {len(mach)}, human {len(hum)})")
rows = []
def add(src, eval_set, texts, attack, level):
    for r, t in zip(src.itertuples(), texts):
        rows.append({"doc_id": f"{r.doc_id}|{eval_set}", "orig_doc_id": r.doc_id, "eval_set": eval_set, "attack": attack,
                     "level": level, "lang": r.lang, "label": r.label, "family": r.family, "generator": r.generator, "text": t})

## T3 / T4 (deterministic, CPU) and defence variants

In [ ]:
for p in A["T3_homoglyph"]["levels"]:
    tag = f"T3_h{int(p*100):02d}"
    for src, suffix in ((mach, ""), (hum, "both")):
        att = [homoglyph_attack(t, p, doc_seed(d, 3)) for t, d in zip(src.text, src.doc_id)]
        add(src, tag + suffix, att, "T3_homoglyph", p)
        add(src, tag + suffix + "_def", [unicode_defense(t) for t in att], "T3_homoglyph+defense", p)
for p in A["T4_typo"]["levels"]:
    tag = f"T4_t{int(p*100):02d}"
    for src, suffix in ((mach, ""), (hum, "both")):
        add(src, tag + suffix, [typo_attack(t, p, doc_seed(d, 4)) for t, d in zip(src.text, src.doc_id)], "T4_typo", p)
# defence applied to clean test texts (checks the defence does not hurt clean accuracy)
add(test, "test_def", [unicode_defense(t) for t in test.text], "none+defense", 0.0)
# sanity: defence must exactly undo the homoglyph attack on these data
chk = [unicode_defense(homoglyph_attack(t, 0.15, 1)) == unicode_defense(t) for t in mach.text.head(200)]
log.info(f"defence restores homoglyph-attacked text exactly: {np.mean(chk):.3f}")

## T1 · LLM paraphrase (cached, OpenRouter)

In [ ]:
import requests, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
OR_KEY = userdata.get("OPENROUTER_API_KEY")
cache_p = P["attacks"] / "paraphrase_cache.jsonl"
done = {r["orig_doc_id"]: r for r in read_jsonl(cache_p)}

def paraphrase(r):
    body = {"model": A["T1_paraphrase"]["model"], "temperature": A["T1_paraphrase"]["temperature"],
            "max_tokens": A["T1_paraphrase"]["max_tokens"], "usage": {"include": True},
            "messages": [{"role": "user", "content": A["T1_paraphrase"]["prompt"].format(lang_name=LANG_META[r.lang]["name"], text=r.text)}]}
    for a in range(6):
        try:
            x = requests.post("https://openrouter.ai/api/v1/chat/completions", json=body, timeout=120,
                              headers={"Authorization": f"Bearer {OR_KEY}"})
            if x.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** a); continue
            x.raise_for_status(); j = x.json()
            return {"orig_doc_id": r.doc_id, "text": j["choices"][0]["message"]["content"], "provider": j.get("provider"),
                    "usage": j.get("usage"), "time_utc": utcnow()}
        except requests.RequestException:
            time.sleep(2 ** a)
    raise RuntimeError("paraphrase failed")

todo = [r for r in mach.itertuples() if r.doc_id not in done]
log.info(f"T1 paraphrases to request: {len(todo)}")
buf = []
with ThreadPoolExecutor(8) as ex:
    futs = [ex.submit(paraphrase, r) for r in todo]
    for i, f in enumerate(as_completed(futs), 1):
        try:
            buf.append(f.result())
        except Exception as e:
            log.warning(e)
        if len(buf) >= 50 or i == len(futs):
            append_jsonl(cache_p, buf); buf = []
done = {r["orig_doc_id"]: r for r in read_jsonl(cache_p)}
log.info(f"T1 cost ≈ ${sum(float((r.get('usage') or {}).get('cost') or 0) for r in done.values()):.2f}")
m1 = mach[mach.doc_id.isin(done)]
add(m1, "T1_para", [truncate_words(normalize_text(done[d]["text"]), D["max_words"]) for d in m1.doc_id], "T1_paraphrase", 1.0)

## T2 · Back-translation with NLLB-200 (sentence-level, cached)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
bt_cache_p = P["attacks"] / "backtranslation_cache.jsonl"
bt_done = {r["orig_doc_id"]: r["text"] for r in read_jsonl(bt_cache_p)}
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(A["T2_backtranslation"]["model"])
mt = AutoModelForSeq2SeqLM.from_pretrained(A["T2_backtranslation"]["model"]).to(dev).eval()
if dev == "cuda":
    mt = mt.half()

def translate(sents, src, tgt, bs=32):
    out = []
    tok.src_lang = src
    for i in range(0, len(sents), bs):
        enc = tok(sents[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=256).to(dev)
        with torch.no_grad():
            gen = mt.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

for lang in LANGS:
    sub = mach[(mach.lang == lang) & (~mach.doc_id.isin(bt_done))]
    if sub.empty:
        continue
    src = LANG_META[lang]["nllb"]; piv = A["T2_backtranslation"]["pivot"].get(lang, A["T2_backtranslation"]["pivot"]["default"])
    sents, owner = [], []
    for r in sub.itertuples():
        ss = [s for s in _SENT_END.split(r.text) if s.strip()]
        sents += ss; owner += [r.doc_id] * len(ss)
    back = translate(translate(sents, src, piv), piv, src)
    res = {}
    for d, s in zip(owner, back):
        res.setdefault(d, []).append(s)
    append_jsonl(bt_cache_p, [{"orig_doc_id": d, "text": " ".join(v), "pivot": piv} for d, v in res.items()])
    log.info(f"[{lang}] back-translated {len(res)} docs via {piv}")
bt_done = {r["orig_doc_id"]: r["text"] for r in read_jsonl(bt_cache_p)}
m2 = mach[mach.doc_id.isin(bt_done)]
add(m2, "T2_bt", [truncate_words(normalize_text(bt_done[d]), D["max_words"]) for d in m2.doc_id], "T2_backtranslation", 1.0)
del mt; gc.collect(); torch.cuda.empty_cache()

## Quality control: meaning preservation (LaBSE) and language (GlotLID)

In [ ]:
att = pd.DataFrame(rows)
from sentence_transformers import SentenceTransformer
labse = SentenceTransformer(A["similarity_model"], device=dev)
orig_text = dict(zip(test.doc_id, test.text))
q = att[att.attack.isin(["T1_paraphrase", "T2_backtranslation", "T3_homoglyph", "T4_typo"])].copy()
e1 = labse.encode([orig_text[d] for d in q.orig_doc_id], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
e2 = labse.encode(q.text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
q["labse_sim"] = (e1 * e2).sum(1)
lid = langid(q.text.tolist())
q["lid_ok"] = [l == LANG_META[g]["lid"] for (l, _), g in zip(lid, q.lang)]
att = att.merge(q[["doc_id", "labse_sim", "lid_ok"]], on="doc_id", how="left")
qual = q.groupby(["eval_set", "lang"]).agg(n=("doc_id", "size"), labse_mean=("labse_sim", "mean"),
                                          labse_p10=("labse_sim", lambda s: s.quantile(0.1)), lid_ok=("lid_ok", "mean")).reset_index()
qual.to_csv(P["attacks"] / "attack_quality.csv", index=False)
save_parquet(att, P["attacks"] / f"attacks_{CFG['data_version']}.parquet")
log.info(f"attack rows: {len(att)}; eval sets: {sorted(att.eval_set.unique())}")
qual

**Note for analysis:** attacked texts are *not* filtered by quality (filtering would bias robustness upward). `labse_sim` and `lid_ok` are stored per document and summarised in `attack_quality.csv`; if T1/T2 quality turns out low for a language, a filtered sensitivity analysis will be added at the analysis stage.